In [9]:
import numpy as np
from scipy.optimize import linprog
import math
from functools import lru_cache

### Задача 1 (инвестпортфель, ЛП)

Дано (доходности):
G1 15%, G2 10%, G3 16%, I1 10%, I2 13%, M 8%
Бюджет: 75 000
Ограничения:

в один фонд ≤ 50% (≤ 37 500)

сумма инвестиций в растущие (G1–G3) >= 2 x суммы инвестиций в облигационные (I1, I2, M)

Оптимум (макс. ожидаемый доход):

G1 = 37 500

G3 = 37 500

остальные = 0
Ожидаемая годовая доходность в деньгах: 11 625.

In [10]:
funds = ["G1", "G2", "G3", "I1", "I2", "M"]
r = np.array([0.15, 0.10, 0.16, 0.10, 0.13, 0.08])
total = 75000
ub = 0.5 * total  # 50% в один фонд

# linprog минимизирует => минимизируем минус доходность
c = -r

# Ограничение growth >= 2*bonds  =>  -growth + 2*bonds <= 0
A_ub = np.array([[-1, -1, -1,  2,  2,  2]], dtype=float)
b_ub = np.array([0.0])

# Сумма инвестиций = total
A_eq = np.ones((1, len(funds)))
b_eq = np.array([total], dtype=float)

bounds = [(0, ub)] * len(funds)

res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
x = res.x
max_profit = -res.fun

print("Статус:", res.message)
for name, val in zip(funds, x):
    print(f"{name}: {val:.2f}")
print("Ожидаемый годовой доход (в деньгах):", max_profit)

Статус: Optimization terminated successfully. (HiGHS Status 7: Optimal)
G1: 37500.00
G2: 0.00
G3: 37500.00
I1: 0.00
I2: -0.00
M: 0.00
Ожидаемый годовой доход (в деньгах): 11625.0


### Задача 2 (график медсестёр, минимизация затрат)

Потребность по дням:
Вс 16, Пн 15, Вт 12, Ср 14, Чт 15, Пт 18, Сб 19.
Каждая медсестра: 5 дней подряд работает + 2 выходных, стоимость 12 000 руб/нед. Нужно минимизировать суммарные затраты => минимизировать число медсестёр.

Что получилось (оптимально)

Минимум медсестёр: 22
Стоимость: 22 × 12 000 = 264 000 руб/нед

Сколько медсестёр начинают 5-дневный цикл в каждый день:

Вс: 0

Пн: 3

Вт: 2

Ср: 5

Чт: 5

Пт: 3

Сб: 4

Проверка покрытия:
Вс 17 (>=16), Пн 15, Вт 12, Ср 14, Чт 15, Пт 18, Сб 19.

In [12]:
days = ["Вс", "Пн", "Вт", "Ср", "Чт", "Пт", "Сб"]
req = np.array([16, 15, 12, 14, 15, 18, 19], dtype=int)

# cov[t,d] = 1 если медсестра, стартующая в день d, работает в день t
# (5 подряд: d, d+1, d+2, d+3, d+4 по модулю 7)
cov = np.zeros((7, 7), dtype=int)
for d in range(7):
    for k in range(5):
        cov[(d + k) % 7, d] = 1

def search_fixed_total(S: int):
    @lru_cache(None)
    def rec(i, remaining, *cov_vec):
        cov_now = np.array(cov_vec, dtype=int)

        if i == 7:
            if remaining == 0 and np.all(cov_now >= req):
                return ()
            return None

        for t in range(7):
            if cov_now[t] >= req[t]:
                continue
            if not any(cov[t, d] == 1 for d in range(i, 7)):
                return None
            if cov_now[t] + remaining < req[t]:
                return None

        for si in range(remaining + 1):
            new_cov = cov_now + si * cov[:, i]
            ans = rec(i + 1, remaining - si, *tuple(new_cov))
            if ans is not None:
                return (si,) + ans
        return None

    return rec(0, S, 0, 0, 0, 0, 0, 0, 0)

def find_optimal():
    lower = math.ceil(req.sum() / 5)
    for S in range(lower, 200):
        sol = search_fixed_total(S)
        if sol is not None:
            return S, np.array(sol, dtype=int)

S, starters = find_optimal()
coverage = cov @ starters
cost = S * 12000

print("Минимум медсестёр:", S)
print("Стоимость (руб/нед):", cost)
print("\nСтартуют (начинают 5-дневный цикл) по дням:")
for d, v in enumerate(starters):
    print(f"{days[d]}: {v}")

print("\nПокрытие по дням (должно быть >= потребности):")
for d in range(7):
    print(f"{days[d]}: нужно {req[d]}, есть {coverage[d]}")

Минимум медсестёр: 22
Стоимость (руб/нед): 264000

Стартуют (начинают 5-дневный цикл) по дням:
Вс: 0
Пн: 3
Вт: 2
Ср: 5
Чт: 5
Пт: 3
Сб: 4

Покрытие по дням (должно быть >= потребности):
Вс: нужно 16, есть 17
Пн: нужно 15, есть 15
Вт: нужно 12, есть 12
Ср: нужно 14, есть 14
Чт: нужно 15, есть 15
Пт: нужно 18, есть 18
Сб: нужно 19, есть 19
